In [12]:
import pandas as pd
import joblib
import numpy as np

In [13]:
df = pd.read_csv("../data/processed/telco_featured.csv")

pipeline = joblib.load("../models/churn_pipeline.pkl")
threshold = joblib.load("../models/threshold.pkl")

X = df.drop("Churn", axis=1)

In [14]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn', 'tenure_group_12-24',
       'tenure_group_24-48', 'tenure_group_48+', 'service_count',
       'high_monthly_charges', 'auto_payment', 'fiber_customer',
       'long_contract', 'customer_lifetime_value', 'avg_monthly_spend',
       'support_dependency'],
      dtype='object')

In [15]:
df["churn_probability"] = pipeline.predict_proba(X)[:, 1]

In [16]:
df["target"] = df["churn_probability"] >= threshold
target_df = df[df["target"] == True]

In [17]:
RETENTION_COST = 500
RETENTION_SUCCESS_RATE = 0.3   # realistic assumption

In [24]:
df["customer_value"] = df["MonthlyCharges"] * df["tenure"]

In [25]:
thresholds = np.linspace(0.1, 0.9, 20)

results = []

for t in thresholds:
    df["target"] = df["churn_probability"] >= t
    target_df = df[df["target"]]
    
    customers_targeted = len(target_df)
    true_positives = target_df[target_df["Churn"] == 1].shape[0]
    
    cost = customers_targeted * RETENTION_COST
    revenue = true_positives * df["customer_value"].mean() * RETENTION_SUCCESS_RATE
    
    roi = (revenue - cost) / cost if cost > 0 else 0
    
    results.append((t, customers_targeted, true_positives, roi))

roi_df = pd.DataFrame(results, columns=["threshold", "targeted", "true_positives", "roi"])
roi_df.sort_values(by="roi", ascending=False)

,threshold,targeted,true_positives,roi
19,0.900000,331,286,0.185446
18,0.857895,580,458,0.083381
17,0.815789,829,612,0.012839
16,0.773684,1166,823,-0.031622
15,0.731579,1468,994,-0.071025
14,0.689474,1752,1111,-0.129991
13,0.647368,2012,1219,-0.168773
12,0.605263,2266,1314,-0.204429
11,0.563158,2486,1394,-0.230683
10,0.521053,2730,1462,-0.265269


In [26]:
best_threshold = roi_df.sort_values(by="roi", ascending=False).iloc[0]["threshold"]
print(f"Best Threshold: {best_threshold}")

Best Threshold: 0.9


In [27]:
df["target"] = df["churn_probability"] >= best_threshold
target_df = df[df["target"]]

In [28]:
retention_rates = [0.2, 0.3, 0.4, 0.5]
costs = [200, 300, 500]

results = []

for rate in retention_rates:
    for cost_per_customer in costs:
        cost = customers_targeted * cost_per_customer
        revenue = true_positives * df["customer_value"].mean() * rate
        
        roi = (revenue - cost) / cost
        
        results.append((rate, cost_per_customer, roi))

sensitivity_df = pd.DataFrame(results, columns=["retention_rate", "cost", "roi"])
sensitivity_df.sort_values(by="roi", ascending=False)

print(f"Customers Targeted: {customers_targeted}")
print(f"True Positives: {true_positives}")
print(f"Total Cost: {cost}")
print(f"Revenue Saved: {revenue_saved:.2f}")
print(f"ROI: {roi:.2f}")

Customers Targeted: 331
True Positives: 286
Total Cost: 165500
Revenue Saved: 66771.51
ROI: 0.98


In [23]:
high_value = target_df[target_df["customer_value"] > df["customer_value"].median()]

print(f"High Value Customers to Target: {len(high_value)}")

High Value Customers to Target: 274


In [29]:
print(f"Optimal Strategy:")
print(f"- Target Customers: {customers_targeted}")
print(f"- Expected ROI: {roi:.2f}")
print(f"- Focus: High-risk, high-value customers")

Optimal Strategy:
- Target Customers: 331
- Expected ROI: 0.98
- Focus: High-risk, high-value customers
